# sirl

In [4]:
from utils import *
from sirl import *
from fpe import *
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.utils.input_utils import transform_input
import numpy as np
import torch
import random
import json

OUTFILE = "061826_sirl_l2_results.json"

# --- Seeds ---
# Fixed across the whole experiment: ensures the FPE eval set is identical
# for every SIRL model we evaluate. NEVER change this between conditions.
FPE_SPLIT_SEED = 42

# SIRL training seeds: vary across these to get error bars on FPE.
SIRL_SEEDS = [0, 1, 2, 3, 4]
n_queries = 1000

# JacoRobot input transform: drops rotation-matrix (63) + joint-angle (7) dims,
# leaving the 27 xyz position dims per state. (N,21,97) -> (N,21,27).
# Both features (table, laptop) are positional, so the dropped dims are distractors.
INPUT_DICT = {'lowdim': False, 'EErot': False, 'noxyz': False, 'norot': True, 'noangles': True}


def tx(trajs):
    """Apply the JacoRobot input transform to a (N, 21, 97) trajectory array."""
    x = torch.as_tensor(trajs, dtype=torch.float32)
    return transform_input(x, INPUT_DICT).numpy()


def set_all_seeds(seed):
    """Set seeds for every source of randomness in SIRL training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- Build FPE eval data ONCE, outside the seed loop ---
# Uses FPE_SPLIT_SEED so the split is deterministic and shared.
train_trajs, train_features, test_trajs, test_features = \
    get_train_test_trajs_feats(seed=FPE_SPLIT_SEED)

# Transform the eval trajectories (features are unchanged — they're labels).
train_trajs = tx(train_trajs)
test_trajs = tx(test_trajs)


results = {}
print(f"\n=== TRAINING SIRL FOR {n_queries} QUERIES ===")
anchors, positives, negatives = load_data(n_queries)

# Transform the training triplets identically.
anchors, positives, negatives = tx(anchors), tx(positives), tx(negatives)

fpes_for_n = []
for seed in SIRL_SEEDS:
    print(f"  -- seed {seed} --")
    set_all_seeds(seed)

    model, _ = train_sirl(
        anchors, positives, negatives,
        use_symmetric_loss=True,
        l2_weight=1/30
    )

    # fpe, probe = eval_fpe_probe(model, train_trajs, train_features, test_trajs, test_features)
    fpe = eval_fpe(model, train_trajs, train_features, test_trajs, test_features)

    # print(f"     FPE = {fpe:.4f}")
    print(f"     FPE = {fpe:.4f}")
    Z = model(torch.as_tensor(test_trajs, dtype=torch.float32, device='cpu')).detach().numpy()
    print(np.linalg.norm(Z, axis=1).mean(), np.linalg.cond(Z))
    fpes_for_n.append(fpe)

fpes = np.array(fpes_for_n)
results[n_queries] = {
    "mean": float(fpes.mean()),
    "std":  float(fpes.std(ddof=1)),
    "all":  fpes.tolist(),
    "seeds": SIRL_SEEDS,
}
print(f"  N={n_queries}: FPE = {fpes.mean():.4f} ± {fpes.std(ddof=1):.4f}")

with open(OUTFILE, "w") as f:
    json.dump(results, f, indent=4)

all_trajs shape: (10000, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elem

In [2]:
s = np.linalg.svd(Z - Z.mean(0), compute_uv=False)
print(s / s[0])

[1.         0.66790426 0.5014861  0.48039347 0.0075514  0.00160271]


In [2]:
with torch.no_grad():
    A = torch.as_tensor(anchors,   dtype=torch.float32)
    P = torch.as_tensor(positives, dtype=torch.float32)
    Ng = torch.as_tensor(negatives, dtype=torch.float32)
    Z = model(torch.as_tensor(test_trajs, dtype=torch.float32)).cpu().numpy()
    ae, pe, ne = model(A), model(P), model(Ng)
    ap = torch.norm(ae-pe, dim=1); an = torch.norm(ae-ne, dim=1)
print("emb norm:", np.linalg.norm(Z,axis=1).mean(),
      "| d(a,p):", ap.mean().item(), "d(a,n):", an.mean().item(),
      "| gap:", (an-ap).mean().item(), "vs margin 0.1")

emb norm: 19.221981 | d(a,p): 0.24023105204105377 d(a,n): 0.3593285381793976 | gap: 0.1190975159406662 vs margin 0.1


In [3]:
print(f"probe train loss {probe.last_train_loss}")
print(f"probe test loss {probe.last_test_loss}")

probe train loss 0.01547785275274221
probe test loss 0.014897217682275664


# random baseline

In [ ]:
from utils import *
from sirl import *
from fpe import *
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.utils.input_utils import transform_input
import numpy as np
import torch
import random
import json

OUTFILE = "061826_random_results.json"

# --- Seeds ---
# Fixed across the whole experiment: ensures the FPE eval set is identical
# for every SIRL model we evaluate. NEVER change this between conditions.
FPE_SPLIT_SEED = 42

# SIRL training seeds: vary across these to get error bars on FPE.
SIRL_SEEDS = [0, 1, 2, 3, 4]
n_queries = 1000

# JacoRobot input transform: drops rotation-matrix (63) + joint-angle (7) dims,
# leaving the 27 xyz position dims per state. (N,21,97) -> (N,21,27).
# Both features (table, laptop) are positional, so the dropped dims are distractors.
INPUT_DICT = {'lowdim': False, 'EErot': False, 'noxyz': False, 'norot': True, 'noangles': True}


def tx(trajs):
    """Apply the JacoRobot input transform to a (N, 21, 97) trajectory array."""
    x = torch.as_tensor(trajs, dtype=torch.float32)
    return transform_input(x, INPUT_DICT).numpy()


def set_all_seeds(seed):
    """Set seeds for every source of randomness in SIRL training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- Build FPE eval data ONCE, outside the seed loop ---
# Uses FPE_SPLIT_SEED so the split is deterministic and shared.
train_trajs, train_features, test_trajs, test_features = \
    get_train_test_trajs_feats(seed=FPE_SPLIT_SEED)

# Transform the eval trajectories (features are unchanged — they're labels).
train_trajs = tx(train_trajs)
test_trajs = tx(test_trajs)


results = {}
print(f"\n=== TRAINING SIRL FOR {n_queries} QUERIES ===")
anchors, positives, negatives = load_data(n_queries)

# Transform the training triplets identically.
anchors, positives, negatives = tx(anchors), tx(positives), tx(negatives)

fpes_for_n = []
for seed in SIRL_SEEDS:
    print(f"  -- seed {seed} --")
    set_all_seeds(seed)

    input_dim = 567
    latent_dim=6
    hidden_dim=1024
    model = SIRL(input_dim=input_dim, hidden_dim=hidden_dim, latent_dim=latent_dim)

    # fpe, probe = eval_fpe_probe(model, train_trajs, train_features, test_trajs, test_features)
    fpe = eval_fpe(model, train_trajs, train_features, test_trajs, test_features)

    # print(f"     FPE = {fpe:.4f}")
    print(f"     FPE = {fpe:.4f}")
    Z = model(torch.as_tensor(test_trajs, dtype=torch.float32, device='cpu')).detach().numpy()
    print(np.linalg.norm(Z, axis=1).mean(), np.linalg.cond(Z))
    fpes_for_n.append(fpe)

fpes = np.array(fpes_for_n)
results[n_queries] = {
    "mean": float(fpes.mean()),
    "std":  float(fpes.std(ddof=1)),
    "all":  fpes.tolist(),
    "seeds": SIRL_SEEDS,
}
print(f"  N={n_queries}: FPE = {fpes.mean():.4f} ± {fpes.std(ddof=1):.4f}")

with open(OUTFILE, "w") as f:
    json.dump(results, f, indent=4)

all_trajs shape: (10000, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elem

In [10]:
from sklearn.linear_model import LinearRegression
Xtr = train_trajs.reshape(len(train_trajs), -1)
Xte = test_trajs.reshape(len(test_trajs), -1)
reg = LinearRegression().fit(Xtr, train_features)
print(np.mean(np.sum((reg.predict(Xte) - test_features)**2, axis=1)))

0.012562515793610224


# tversky sirl

In [6]:
from utils import *
from tversky_sirl import *
from fpe import *
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.utils.input_utils import transform_input
import numpy as np
import torch
import random
import json

OUTFILE = "061826_tversky_sirl_results.json"

# --- Seeds ---
# Fixed across the whole experiment: ensures the FPE eval set is identical
# for every SIRL model we evaluate. NEVER change this between conditions.
FPE_SPLIT_SEED = 42

# SIRL training seeds: vary across these to get error bars on FPE.
SIRL_SEEDS = [0, 1, 2, 3, 4]
n_queries = 1000

# JacoRobot input transform: drops rotation-matrix (63) + joint-angle (7) dims,
# leaving the 27 xyz position dims per state. (N,21,97) -> (N,21,27).
# Both features (table, laptop) are positional, so the dropped dims are distractors.
INPUT_DICT = {'lowdim': False, 'EErot': False, 'noxyz': False, 'norot': True, 'noangles': True}


def tx(trajs):
    """Apply the JacoRobot input transform to a (N, 21, 97) trajectory array."""
    x = torch.as_tensor(trajs, dtype=torch.float32)
    return transform_input(x, INPUT_DICT).numpy()


def set_all_seeds(seed):
    """Set seeds for every source of randomness in SIRL training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- Build FPE eval data ONCE, outside the seed loop ---
# Uses FPE_SPLIT_SEED so the split is deterministic and shared.
train_trajs, train_features, test_trajs, test_features = \
    get_train_test_trajs_feats(seed=FPE_SPLIT_SEED)

# Transform the eval trajectories (features are unchanged — they're labels).
train_trajs = tx(train_trajs)
test_trajs = tx(test_trajs)


results = {}
print(f"\n=== TRAINING SIRL FOR {n_queries} QUERIES ===")
anchors, positives, negatives = load_data(n_queries)

# Transform the training triplets identically.
anchors, positives, negatives = tx(anchors), tx(positives), tx(negatives)

fpes_for_n = []
for seed in SIRL_SEEDS:
    print(f"  -- seed {seed} --")
    set_all_seeds(seed)

    model, _ = train_tversky_sirl(
        anchors, positives, negatives,
        use_symmetric_loss=True,
        # l2_weight=1/30
    )

    # fpe, probe = eval_fpe_probe(model, train_trajs, train_features, test_trajs, test_features)
    fpe = eval_fpe(model, train_trajs, train_features, test_trajs, test_features)

    # print(f"     FPE = {fpe:.4f}")
    print(f"     FPE = {fpe:.4f}")
    Z = model(torch.as_tensor(test_trajs, dtype=torch.float32, device='cpu')).detach().numpy()
    print(np.linalg.norm(Z, axis=1).mean(), np.linalg.cond(Z))
    fpes_for_n.append(fpe)

fpes = np.array(fpes_for_n)
results[n_queries] = {
    "mean": float(fpes.mean()),
    "std":  float(fpes.std(ddof=1)),
    "all":  fpes.tolist(),
    "seeds": SIRL_SEEDS,
}
print(f"  N={n_queries}: FPE = {fpes.mean():.4f} ± {fpes.std(ddof=1):.4f}")

with open(OUTFILE, "w") as f:
    json.dump(results, f, indent=4)

all_trajs shape: (10000, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elem

# pca

In [ ]:
from utils import *
from tversky_sirl import *
from fpe import *
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.utils.input_utils import transform_input
import numpy as np
import torch
import random
import json

OUTFILE = "061826_pca_results.json"

# --- Seeds ---
# Fixed across the whole experiment: ensures the FPE eval set is identical
# for every SIRL model we evaluate. NEVER change this between conditions.
FPE_SPLIT_SEED = 42

# SIRL training seeds: vary across these to get error bars on FPE.
SIRL_SEEDS = [0, 1, 2, 3, 4]
n_queries = 1000

# JacoRobot input transform: drops rotation-matrix (63) + joint-angle (7) dims,
# leaving the 27 xyz position dims per state. (N,21,97) -> (N,21,27).
# Both features (table, laptop) are positional, so the dropped dims are distractors.
INPUT_DICT = {'lowdim': False, 'EErot': False, 'noxyz': False, 'norot': True, 'noangles': True}


def tx(trajs):
    """Apply the JacoRobot input transform to a (N, 21, 97) trajectory array."""
    x = torch.as_tensor(trajs, dtype=torch.float32)
    return transform_input(x, INPUT_DICT).numpy()


def set_all_seeds(seed):
    """Set seeds for every source of randomness in SIRL training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- Build FPE eval data ONCE, outside the seed loop ---
# Uses FPE_SPLIT_SEED so the split is deterministic and shared.
train_trajs, train_features, test_trajs, test_features = \
    get_train_test_trajs_feats(seed=FPE_SPLIT_SEED)

# Transform the eval trajectories (features are unchanged — they're labels).
train_trajs = tx(train_trajs)
test_trajs = tx(test_trajs)


results = {}
print(f"\n=== TRAINING SIRL FOR {n_queries} QUERIES ===")
anchors, positives, negatives = load_data(n_queries)

# Transform the training triplets identically.
anchors, positives, negatives = tx(anchors), tx(positives), tx(negatives)

fpes_for_n = []
for seed in SIRL_SEEDS:
    print(f"  -- seed {seed} --")
    set_all_seeds(seed)

    query_trajs = np.concatenate([anchors, positives, negatives], axis=0)
    pca = fit_pca(query_trajs, n_components=6, random_state=seed)

    fpe = eval_pca_fpe(pca, train_trajs, train_features, test_trajs, test_features)

    # fpe, probe = eval_fpe_probe(model, train_trajs, train_features, test_trajs, test_features)
    fpe = eval_fpe(model, train_trajs, train_features, test_trajs, test_features)

    # print(f"     FPE = {fpe:.4f}")
    print(f"     FPE = {fpe:.4f}")
    Z = model(torch.as_tensor(test_trajs, dtype=torch.float32, device='cpu')).detach().numpy()
    print(np.linalg.norm(Z, axis=1).mean(), np.linalg.cond(Z))
    fpes_for_n.append(fpe)

fpes = np.array(fpes_for_n)
results[n_queries] = {
    "mean": float(fpes.mean()),
    "std":  float(fpes.std(ddof=1)),
    "all":  fpes.tolist(),
    "seeds": SIRL_SEEDS,
}
print(f"  N={n_queries}: FPE = {fpes.mean():.4f} ± {fpes.std(ddof=1):.4f}")

with open(OUTFILE, "w") as f:
    json.dump(results, f, indent=4)

all_trajs shape: (10000, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elem

NameError: name 'query_trajs' is not defined